# Crossword-Generator Dataset — OpenEvolve Run

This notebook runs **only the data-generation step**: serve Qwen3-4B as the search model, evolve crossword-generator programs with OpenEvolve against our deterministic scorer, and harvest the trace into a SOAR-style `(spec -> program)` dataset. QLoRA training + base-vs-tuned eval come in a separate notebook.

**Runtime:** GPU (T4 works; A100 faster). **Order:** run cells top to bottom.

## 1. Get the code
Clone the repo (set your URL) or upload the project folder and set `PROJECT_DIR`.

In [ ]:
REPO_URL = "https://github.com/YOUR_USER/YOUR_REPO.git"   # <-- set this, or upload the folder
import os
!git clone -q $REPO_URL slm || echo "clone skipped/failed"
PROJECT_DIR = "/content/slm"   # adjust if you uploaded elsewhere
assert os.path.isdir(os.path.join(PROJECT_DIR, "pipeline")), "Set PROJECT_DIR to the repo root"
%cd $PROJECT_DIR

## 2. Install dependencies
Our pipeline is pure-Python stdlib; the word lists ship in `data/`.

In [ ]:
!pip -q install openevolve vllm requests

## 3. Confirm the OpenEvolve API (catch version drift)
We built `pipeline/oe_evaluator.py` and `pipeline/run_openevolve.py` against the documented API. This cell checks the evaluator import and the CLI entry point. If either differs, update `oe_evaluator.py`'s `EvaluationResult` import and `run_openevolve.py`'s `_run_openevolve_cli`.

In [ ]:
import openevolve
print("openevolve version:", getattr(openevolve, "__version__", "?"))
try:
    from openevolve.evaluation_result import EvaluationResult
    print("EvaluationResult import OK")
except Exception as e:
    print("EvaluationResult import DIFFERS ->", e)
print("top-level names:", [n for n in dir(openevolve) if not n.startswith("_")])
!python -m openevolve.cli --help 2>&1 | head -n 15 || echo "no openevolve.cli — check the CLI entry point"

## 4. Serve Qwen3-4B via vLLM (the search model)
T4: keep `--dtype half`. A100/L4: you may use `bfloat16` and a larger `--max-model-len`.

In [ ]:
import subprocess, sys, time, requests
MODEL = "Qwen/Qwen3-4B"
log = open("vllm.log", "w")
server = subprocess.Popen(
    [sys.executable, "-m", "vllm.entrypoints.openai.api_server",
     "--model", MODEL, "--dtype", "half", "--max-model-len", "8192",
     "--gpu-memory-utilization", "0.90", "--port", "8000"],
    stdout=log, stderr=subprocess.STDOUT)
up = False
for _ in range(120):
    try:
        if requests.get("http://localhost:8000/v1/models", timeout=2).ok:
            up = True; break
    except Exception:
        pass
    time.sleep(10)
print("vLLM up" if up else "NOT up — see vllm.log below")
!tail -n 20 vllm.log

## 5. Run OpenEvolve across the pilot specs
The driver seeds OpenEvolve with **three distinct generator families** — `reference_v1` (greedy + backtracking), `csp_ac3` (CSP + AC-3 propagation), and `beam_search` (greedy beam) — running one evolution per (seed x spec) and merging every evaluated candidate into a single harvest. Multiple seeds give evolution diverse starting points and a more varied dataset.

Pilot uses sizes **7,9** (all seeds bootstrap a valid grid there). Each candidate is scored by our sandbox+scorer and appended to `runs/pilot/harvest.jsonl`; the driver then builds `runs/pilot/dataset/{train,dev,test}.jsonl`. Start small, then scale `--n-specs` / `--iterations`.

In [ ]:
!python pipeline/run_openevolve.py \
    --seeds reference_v1,csp_ac3,beam_search \
    --sizes 7,9 --n-specs 8 --iterations 60 \
    --model Qwen/Qwen3-4B --api-base http://localhost:8000/v1 \
    --out runs/pilot

## 6. Inspect the harvested dataset

In [ ]:
import json, os
h = [json.loads(l) for l in open("runs/pilot/harvest.jsonl")]
print("candidates harvested:", len(h))
for split in ("train", "dev", "test"):
    p = f"runs/pilot/dataset/{split}.jsonl"
    n = sum(1 for _ in open(p)) if os.path.exists(p) else 0
    print(f"  {split}: {n}")
tp = "runs/pilot/dataset/train.jsonl"
if os.path.exists(tp) and os.path.getsize(tp):
    ex = json.loads(open(tp).readline())
    print("\n--- sample SPEC ---\n", ex["messages"][1]["content"][:300])
    print("\n--- sample PROGRAM (head) ---\n", ex["messages"][2]["content"][:400])

## 7. Save outputs to Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!mkdir -p /content/drive/MyDrive/slm_runs
!cp -r runs/pilot /content/drive/MyDrive/slm_runs/
print("saved runs/pilot to Drive")

## Next
With `dataset/train.jsonl` in hand, the **training notebook** QLoRA-fine-tunes Qwen3-4B on it and runs the base-vs-tuned eval (reusing our sandbox+scorer), then exports a GGUF for local inference.